<a href="https://colab.research.google.com/github/bhumika-231206/MAPIE/blob/master/tensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install git+https://github.com/bhumika-231206/tensordict.git

  Cloning https://github.com/bhumika-231206/tensordict.git to /tmp/pip-req-build-v7i5zczg
  Running command git clone --filter=blob:none --quiet https://github.com/bhumika-231206/tensordict.git /tmp/pip-req-build-v7i5zczg
  Resolved https://github.com/bhumika-231206/tensordict.git to commit 331d77f1c0bf84b7ee499bb4cacb902c924857de
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
from tensordict import TensorDict, lazy_stack

def mk():
    return lazy_stack([
        TensorDict(a=torch.full((3,), float(i)), batch_size=[3])
        for i in range(4)
    ])

def f(ls):
    ls.update_(lazy_stack(ls.tensordicts[::-1]))
    return ls["a"][:, 0]

print("PyTorch:", torch.__version__)
print("Eager:", f(mk()).tolist())

compiled_f = torch.compile(f, fullgraph=True)
print("Inductor:", compiled_f(mk()).tolist())

PyTorch: 2.11.0+cpu
Eager: [3.0, 2.0, 1.0, 0.0]
Inductor: [3.0, 2.0, 2.0, 3.0]


In [4]:
import torch

def make_tensors():
    return [torch.full((3,), float(i)) for i in range(4)]

def foreach_copy(xs):
    sources = [x.clone() for x in xs[::-1]]
    torch._foreach_copy_(xs, sources)
    return torch.stack(xs)[:, 0]

def individual_copy(xs):
    sources = [x.clone() for x in xs[::-1]]
    for dest, src in zip(xs, sources):
        dest.copy_(src)
    return torch.stack(xs)[:, 0]

print("PyTorch:", torch.__version__)

print("\n1. Eager foreach:")
print(foreach_copy(make_tensors()).tolist())

print("\n2. Compiled foreach (Inductor):")
compiled_foreach = torch.compile(foreach_copy, fullgraph=True)
print(compiled_foreach(make_tensors()).tolist())

print("\n3. Compiled individual copy (Inductor):")
compiled_individual = torch.compile(individual_copy, fullgraph=True)
print(compiled_individual(make_tensors()).tolist())

PyTorch: 2.11.0+cpu

1. Eager foreach:
[3.0, 2.0, 1.0, 0.0]

2. Compiled foreach (Inductor):
[3.0, 2.0, 2.0, 3.0]

3. Compiled individual copy (Inductor):
[3.0, 2.0, 1.0, 0.0]
